In [2]:
import torch
import triton
import triton.language as tl


@triton.jit
def add_kernel(x_ptr,  # *Pointer* to first input vector. 指向第一个输入向量的指针。
               y_ptr,  # *Pointer* to second input vector. 指向第二个输入向量的指针。
               output_ptr,  # *Pointer* to output vector. 指向输出向量的指针。
               n_elements,  # Size of the vector. 向量的大小。
               BLOCK_SIZE: tl.constexpr,  # Number of elements each program should process. 每个程序应处理的元素数量。
               # NOTE: `constexpr` so it can be used as a shape value. 注意：`constexpr` 因此它可以用作形状值。
               ):
    # There are multiple 'programs' processing different data. We identify which program
    # 有多个“程序”处理不同的数据。需要确定是哪一个程序：
    pid = tl.program_id(axis=0)  # We use a 1D launch grid so axis is 0. 使用 1D 启动网格，因此轴为 0。
    # This program will process inputs that are offset from the initial data.
    # 该程序将处理相对初始数据偏移的输入。
    # For instance, if you had a vector of length 256 and block_size of 64, the programs would each access the elements [0:64, 64:128, 128:192, 192:256].
    # 例如，如果有一个长度为 256, 块大小为 64 的向量，程序将各自访问 [0:64, 64:128, 128:192, 192:256] 的元素。
    # Note that offsets is a list of pointers:
    # 注意 offsets 是指针列表：
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    # Create a mask to guard memory operations against out-of-bounds accesses.
    # 创建掩码以防止内存操作超出边界访问。
    mask = offsets < n_elements
    # Load x and y from DRAM, masking out any extra elements in case the input is not a multiple of the block size.
    # 从 DRAM 加载 x 和 y，如果输入不是块大小的整数倍，则屏蔽掉任何多余的元素。
    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)
    output = x + y
    # Write x + y back to DRAM.
    # 将 x + y 写回 DRAM。
    tl.store(output_ptr + offsets, output, mask=mask)
    
def add(x: torch.Tensor, y: torch.Tensor):
    # We need to preallocate the output.
    # 需要预分配输出。
    output = torch.empty_like(x)
    assert x.is_cuda and y.is_cuda and output.is_cuda
    n_elements = output.numel()
    # The SPMD launch grid denotes the number of kernel instances that run in parallel.
    # SPMD 启动网格表示并行运行的内核实例的数量。
    # It is analogous to CUDA launch grids. It can be either Tuple[int], or Callable(metaparameters) -> Tuple[int].
    # 它类似于 CUDA 启动网格。它可以是 Tuple[int]，也可以是 Callable(metaparameters) -> Tuple[int]。
    # In this case, we use a 1D grid where the size is the number of blocks:
    # 在这种情况下，使用 1D 网格，其中大小是块的数量：
    grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']), )
    # NOTE:
    # 注意：
    #  - Each torch.tensor object is implicitly converted into a pointer to its first element.
    #  - 每个 torch.tensor 对象都会隐式转换为其第一个元素的指针。
    #  - `triton.jit`'ed functions can be indexed with a launch grid to obtain a callable GPU kernel.
    #  - `triton.jit` 函数可以通过启动网格索引来获得可调用的 GPU 内核。
    #  - Don't forget to pass meta-parameters as keywords arguments.
    #  - 不要忘记以关键字参数传递元参数。
    add_kernel[grid](x, y, output, n_elements, BLOCK_SIZE=1024)
    # We return a handle to z but, since `torch.cuda.synchronize()` hasn't been called, the kernel is still running asynchronously at this point.
    # 返回 z 的句柄，但由于 `torch.cuda.synchronize()` 尚未被调用，此时内核仍在异步运行。
    return output

torch.manual_seed(0)
size = 98432
x = torch.rand(size, device='cuda')
y = torch.rand(size, device='cuda')
output_torch = x + y
output_triton = add(x, y)
print(output_torch)
print(output_triton)
print(f'The maximum difference between torch and triton is '
      f'{torch.max(torch.abs(output_torch - output_triton))}')

tensor([1.3713, 1.3076, 0.4940,  ..., 0.7282, 0.4773, 0.0914], device='cuda:0')
tensor([1.3713, 1.3076, 0.4940,  ..., 0.7282, 0.4773, 0.0914], device='cuda:0')
The maximum difference between torch and triton is 0.0


In [3]:
import torch
import triton
import triton.language as tl

# -----------------------------
# Triton kernel (你来填核心逻辑)
# -----------------------------
@triton.jit
def bias_relu_kernel(
    X_ptr, B_ptr, Y_ptr,
    M: tl.constexpr, N: tl.constexpr,
    stride_xm: tl.constexpr, stride_xn: tl.constexpr,
    stride_ym: tl.constexpr, stride_yn: tl.constexpr,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr,
):
    # 2D program id：pid_m 管 M 方向，pid_n 管 N 方向
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    # TODO: 你来完成：
    # 1) 计算当前 program 负责的 (row_offsets, col_offsets)
    row_offsets = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    col_offsets = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    # 2) 生成 X 的指针并 tl.load（带 mask）
    x_ptr = X_ptr + row_offsets[:, None] * stride_xm + col_offsets[None, :] * stride_xn
    x = tl.load(x_ptr, mask=(row_offsets[:, None] < M) & (col_offsets[None, :] < N))
    # 3) 生成 B 的指针（b 按列广播）并 tl.load（带 mask）
    b_ptr = B_ptr + col_offsets 
    b = tl.load(b_ptr, mask=col_offsets < N)
    # 4) 做 x + b，然后 relu
    y = tl.maximum(x + b[None, :], 0)
    # 5) tl.store 到 Y（带 mask）
    y_ptr = Y_ptr + row_offsets[:, None] * stride_ym + col_offsets[None, :] * stride_yn
    tl.store(y_ptr, y, mask=(row_offsets[:, None] < M) & (col_offsets[None, :] < N))


# -----------------------------
# Python wrapper
# -----------------------------
def bias_relu_triton(x: torch.Tensor, b: torch.Tensor, block_m=128, block_n=256):
    """
    x: [M, N]
    b: [N]
    """
    assert x.ndim == 2
    assert b.ndim == 1
    assert x.shape[1] == b.shape[0]
    assert x.is_cuda and b.is_cuda
    assert x.is_contiguous(), "先用 contiguous 简化入门"
    assert b.is_contiguous()

    M, N = x.shape
    y = torch.empty_like(x)

    grid = (triton.cdiv(M, block_m), triton.cdiv(N, block_n))

    bias_relu_kernel[grid](
        x, b, y,
        M=M, N=N,
        stride_xm=x.stride(0), stride_xn=x.stride(1),
        stride_ym=y.stride(0), stride_yn=y.stride(1),
        BLOCK_M=block_m, BLOCK_N=block_n,
        num_warps=4,   # 入门先固定，后面你可以调参
    )
    return y


# -----------------------------
# Reference + tests
# -----------------------------
def ref_bias_relu(x, b):
    return torch.relu(x + b)

def run_correctness_tests(device="cuda", dtype=torch.float16, seed=0):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    test_shapes = [
        (1, 1),
        (2, 17),
        (33, 65),
        (128, 256),
        (257, 511),
        (1024, 1024),
        (999, 1001),
    ]

    for (M, N) in test_shapes:
        x = torch.randn((M, N), device=device, dtype=dtype).contiguous()
        b = torch.randn((N,), device=device, dtype=dtype).contiguous()

        y_ref = ref_bias_relu(x, b)
        y_tri = bias_relu_triton(x, b)

        # 根据 dtype 设定容忍度
        if dtype in (torch.float16, torch.bfloat16):
            atol, rtol = 1e-2, 1e-2
        else:
            atol, rtol = 1e-5, 1e-5

        ok = torch.allclose(y_tri, y_ref, atol=atol, rtol=rtol)
        max_err = (y_tri - y_ref).abs().max().item()
        print(f"[M={M}, N={N}] ok={ok} max_err={max_err:.3e}")
        if not ok:
            raise AssertionError("Correctness failed!")

    print("All correctness tests passed ✅")

def quick_bench(device="cuda", dtype=torch.float16, M=4096, N=4096, iters=100):
    x = torch.randn((M, N), device=device, dtype=dtype).contiguous()
    b = torch.randn((N,), device=device, dtype=dtype).contiguous()

    # warmup
    for _ in range(10):
        _ = bias_relu_triton(x, b)
    torch.cuda.synchronize()

    starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    starter.record()
    for _ in range(iters):
        _ = bias_relu_triton(x, b)
    ender.record()
    torch.cuda.synchronize()
    ms = starter.elapsed_time(ender) / iters

    # 对比 pytorch
    for _ in range(10):
        _ = torch.relu(x + b)
    torch.cuda.synchronize()

    starter.record()
    for _ in range(iters):
        _ = torch.relu(x + b)
    ender.record()
    torch.cuda.synchronize()
    ms_ref = starter.elapsed_time(ender) / iters

    print(f"Triton: {ms:.3f} ms/iter | Torch: {ms_ref:.3f} ms/iter")

if __name__ == "__main__":
    # 先写完 kernel 再跑
    run_correctness_tests(dtype=torch.float16)
    quick_bench(dtype=torch.float16)

[M=1, N=1] ok=True max_err=0.000e+00
[M=2, N=17] ok=True max_err=0.000e+00
[M=33, N=65] ok=True max_err=0.000e+00
[M=128, N=256] ok=True max_err=0.000e+00
[M=257, N=511] ok=True max_err=0.000e+00
[M=1024, N=1024] ok=True max_err=0.000e+00
[M=999, N=1001] ok=True max_err=0.000e+00
All correctness tests passed ✅
Triton: 2.840 ms/iter | Torch: 0.574 ms/iter
